In [ ]:
import pandas as pd
import torchvision.transforms as T
import numpy as np
import random
import torch
import torch.nn as nn
import datetime
import time
import os
import json
from transformers import get_linear_schedule_with_warmup
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True  # Enforce deterministic algorithms
        torch.backends.cudnn.benchmark = False     # Disable benchmark for reproducibility

    os.environ['PYTHONHASHSEED'] = str(seed)       # Seed Python hashing, which can affect ordering
set_seed(42)


LABEL_COLS = [
    "Vaccum Cleaning", "Mopping the Floor", "Carry Warm Food",
    "Carry Cold Food", "Carry Drinks", "Carry Small Objects",
    "Carry Large Objects", "Cleaning", "Starting a conversation"
]


import sys
sys.path.append('..')

from models.buffers import NaiveRehearsalBuffer
from data_processing.data_utils import get_transform, serialize_transform, get_domain_dataloaders, pool_domain_dataloaders, pool_domain_dataloaders, get_crossvalidation_domain_loaders, IMAGENET_NORM, get_domain_dataloaders_from_hdf5
from models.training_utils import heuristic_dualbranch_batch, unified_train_loop, archive_domain_models
from models.heuristicSplitModel import DualBranchModel, DualBranchModel_fusions

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")    

default_domains = ['Home', 'BigOffice-2', 'BigOffice-3', 'Hallway', 'MeetingRoom', 'SmallOffice']

testing_scenarios = {
    'silh.base': ('base', 60, 60, None, None, "../data/mean_data_pepper.hdf5", [get_transform((144,256), IMAGENET_NORM)]*2, ['image_path_env', 'image_path_soc'], False, nn.MSELoss(), {'branch':'mobilenetv2'}, False, True, default_domains),
    'closeup.base': ('base', 60, 60, None, None, "../data/robotfocus128_pepper_data.hdf5", [get_transform((144,256), IMAGENET_NORM), get_transform((128,128), IMAGENET_NORM)], ['image_path_scene', 'image_path_focus_x2'], False, nn.MSELoss(), {'branch':'mobilenetv2'}, False, True, default_domains),
    'order_complexup.base': ('base', 60, 60, None, None, "../data/mean_data_pepper.hdf5", [get_transform((144,256), IMAGENET_NORM)]*2, ['image_path_env_boxes_plus', 'image_path_soc'], False, nn.MSELoss(), {'branch':'mobilenetv2'}, False, True, ['Hallway', 'SmallOffice', 'MeetingRoom', 'BigOffice-3', 'BigOffice-2', 'Home']),
    'order_complexdown.base': ('base', 60, 60, None, None, "../data/mean_data_pepper.hdf5", [get_transform((144,256), IMAGENET_NORM)]*2, ['image_path_env_boxes_plus', 'image_path_soc'], False, nn.MSELoss(), {'branch':'mobilenetv2'}, False, True, ['Home', 'BigOffice-2', 'BigOffice-3', 'MeetingRoom', 'SmallOffice', 'Hallway']),
    'order_contrA.base': ('base', 60, 60, None, None, "../data/mean_data_pepper.hdf5", [get_transform((144,256), IMAGENET_NORM)]*2, ['image_path_env_boxes_plus', 'image_path_soc'], False, nn.MSELoss(), {'branch':'mobilenetv2'}, False, True, ['BigOffice-3', 'SmallOffice', 'Home', 'Hallway', 'MeetingRoom', 'BigOffice-2']),
    'order_contrB.base': ('base', 60, 60, None, None, "../data/mean_data_pepper.hdf5", [get_transform((144,256), IMAGENET_NORM)]*2, ['image_path_env_boxes_plus', 'image_path_soc'], False, nn.MSELoss(), {'branch':'mobilenetv2'}, False, True, ['MeetingRoom', 'Home', 'Hallway', 'BigOffice-2', 'SmallOffice', 'BigOffice-3']),
}
    
for name, (ablation, epochs, buffer_size, train_val_path, test_path, hdf5_path, transform_list, img_path_cols, scene_as_label, loss_fn, setup, joint_training, use_buffer, domains) in testing_scenarios.items():
        train_df = pd.read_pickle(train_val_path) if train_val_path is not None else None
        test_df = pd.read_pickle(test_path) if test_path is not None else None
        main_hdf5_dataset_path = hdf5_path
        setup=setup #{'branch':'mobilenetv2'}
        seed=42
        set_seed(seed)
        grad_clipping=True
        batch=(32, 64, 64)
        buffer_size=buffer_size
        branch_norm=True
        earlystopping=True
        epochs=epochs
        joint_training=joint_training
        scene_as_label=scene_as_label
        pin_memory=True
        persistent_workers=False
        num_workers=0
        use_buffer=use_buffer


        for fold in range(0,5):
            print(f"\nTesting fold: {fold}")

            hdf5_dataset_path = main_hdf5_dataset_path[:-5] + f'_fold{fold}.hdf5'

            if hdf5_path:
                domain_dataloaders = get_domain_dataloaders_from_hdf5(hdf5_path=hdf5_dataset_path, domains=domains, img_path_cols=img_path_cols, num_workers=num_workers, set_first_element_as_domain_label=scene_as_label, pin_memory=pin_memory, persistent_workers=persistent_workers)
            else:
                domain_dataloaders = get_domain_dataloaders(train_df, seed=seed, batch_sizes=batch, img_path_cols=img_path_cols, transforms=transform_list, num_workers=num_workers, include_test=test_df, set_first_element_as_domain_label=scene_as_label, pin_memory=pin_memory, persistent_workers=persistent_workers)
            
            if joint_training:
                domain_dataloaders = pool_domain_dataloaders(domain_dataloaders)

            print(f"\nTesting: {name} Ablation: {ablation}")

            if ablation == 'nomask':
                setup['ablation'] = 'focus'
            elif ablation == 'onlysoc':
                setup['ablation'] = 'scene'
            elif ablation == 'onlyenv':
                setup['ablation'] = 'focus'
            else:
                setup['ablation'] = False

            if scene_as_label:
                setup['scene'] = 'label'
            dropout_rate=0.3
            model = DualBranchModel(dropout_rate=dropout_rate, setup=setup, branch_norm=branch_norm)
            dual_model = model.to(device)
            trainable_params = [p for p in dual_model.parameters() if p.requires_grad]
            lr=1e-3
            optimizer = torch.optim.Adam(trainable_params, lr=lr)
            buffer = NaiveRehearsalBuffer(buffer_size=buffer_size) if use_buffer else None

            epochs=epochs
            exp_name = f"{name}_{fold=}_{buffer_size=}_{epochs=}_{seed=}_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
            
            scheduler=(get_linear_schedule_with_warmup, 0.05)
            refresh_optimiser=False

            dualbranch_kwargs = {
                    'mse_criterion': loss_fn #nn.MSELoss(),
                }

            config = {
                "timestamp": datetime.datetime.now().strftime('%Y%m%d_%H%M%S'),
                "model": {
                    "name": str(type(model)),
                    "setup": setup,
                    "branch_norm":branch_norm,
                    "dropout_rate":dropout_rate,
                },
                "buffer": str(type(buffer)),
                "buffer_size": buffer_size,
                "optimizer": str(type(optimizer)),
                "learning_rate": lr,
                "scheduler": {
                    "type": "transformers.get_linear_schedule_with_warmup",
                    "warmup": scheduler[1],
                },
                "device": str(device),
                "loss": str(dualbranch_kwargs['mse_criterion']),
                "joint_training": joint_training, #trained on pooled domains, not continual learning
                "seed": seed,
                "domains_order": domains,
                "train_val_df": train_val_path,
                "test_df": test_path,
                "hdf5_dataset_path": hdf5_dataset_path,
                "input_columns": img_path_cols,
                "input_transforms": [serialize_transform(x) for x in transform_list],
                "dataloader": {
                    "batch_size": batch,
                    "num_workers":num_workers,
                    "pin_memory": pin_memory,
                    "persistent_workers": persistent_workers,
                },
                "epochs": epochs,
                "early_stopping": {
                    "patience":15,
                    "delta":1e-3,
                },
                "gradient_clipping": grad_clipping,
                "refresh_optimiser": refresh_optimiser,
            }

            with open(f"../checkpoints/{exp_name}_config.json", "w") as f:
                json.dump(config, f, indent=4)

            unified_train_loop(
                model=dual_model,
                domains=domains if not joint_training else ['joint'],
                domain_dataloaders=domain_dataloaders,
                buffer=buffer,
                optimizer=optimizer,
                device=device,
                batch_fn=heuristic_dualbranch_batch,
                batch_kwargs=dualbranch_kwargs,
                num_epochs=epochs,
                exp_name=exp_name,
                gradient_clipping=grad_clipping,
                checkpoint_dir="../checkpoints",
                validation_set='val',
                scheduler=scheduler,
                refresh_optimiser=refresh_optimiser,
                early_stopping=earlystopping,
            )